## Import data

In [8]:
# import os
# print(os.getcwd())

import pandas as pd 

data = pd.read_excel('../data/feedback_from_paramedics_on_the_road.xlsx', sheet_name='Data')

subset = data.loc[:, ['How easy is it to access a GP via the Non-Public number?',
                     'What services do you need to support you with admission avoidance to ED?',
                     'How likely are you to recommend OPAL+ to a colleague?',
                     'Have you used Patient Services on DoS?',
                     'Do Senior Clinicians come out to the Ambulance to discuss your patient when delayed?',
                     'Who do you call for Clinical Support on scene?',
                     'What do you feel would improve handover delays?',
                    'Have you used SDEC Services?'
                     ]]

print(subset.head())

  How easy is it to access a GP via the Non-Public number?  \
0                                               Easy         
1                                               Easy         
2  The numbers don’t always work, either ring out...         
3                                               Easy         
4  mostly easy however sometimes the number is no...         

  What services do you need to support you with admission avoidance to ED?  \
0  acceptance of referrals / more SDEC availability                          
1                             Mental health services                         
2  Mental health, children in the south bham area...                         
3                       Access/acceptance by SAU/MAU                         
4  UTC/UCC/SDEC appointments more accepted by pat...                         

  How likely are you to recommend OPAL+ to a colleague?  \
0                                                  5      
1                                       

In [20]:
Q1_list = subset['How easy is it to access a GP via the Non-Public number?'].dropna().astype(str).tolist()
Q2_list = subset['What services do you need to support you with admission avoidance to ED?'].dropna().astype(str).tolist()
Q6_list = subset['Who do you call for Clinical Support on scene?'].dropna().astype(str).tolist()
Q7_list = subset['What do you feel would improve handover delays?'].dropna().astype(str).tolist()
Q8_list = subset['Have you used SDEC Services?'].dropna().astype(str).tolist()

In [24]:
print(Q1_list[:5])

['Easy', 'Easy', 'The numbers don’t always work, either ring out or say not known. When they do work they are brilliant ', 'Easy', 'mostly easy however sometimes the number is not answered especially when its a mobile phone.']


## Sentence embedding model

In [36]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired
import nbformat
from hdbscan import HDBSCAN

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# This is unnessary as BERTopic will automatically encode the documents using the specified embedding model, but we can do it manually if we want to inspect the embeddings or use them for other purposes.
embeddings_Q1 = model.encode(Q1_list, show_progress_bar=True)
embeddings_Q2 = model.encode(Q2_list, show_progress_bar=True)
embeddings_Q6 = model.encode(Q6_list, show_progress_bar=True)
embeddings_Q7 = model.encode(Q7_list, show_progress_bar=True)
embeddings_Q8 = model.encode(Q8_list, show_progress_bar=True)


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.33s/it]


In [ ]:
# Inpect the shape and type of the embeddings
print(embeddings_Q1.shape)
print(type(embeddings_Q1))
print(embeddings_Q1[0]) # vector representation of the first response in Q1_list
print(Q1_list[0]) # the original text of the first response in Q1_list


(77, 384)
<class 'numpy.ndarray'>
[-8.00802279e-03  6.37836680e-02 -4.10496369e-02  6.34275079e-02
 -9.10963921e-04 -1.63701887e-03  2.69002672e-02  2.96350243e-03
 -6.16199672e-02  2.52194293e-02 -4.29989621e-02 -8.04116279e-02
 -1.43728415e-02  5.58779426e-02  9.23129171e-03 -8.42367206e-03
  1.09113734e-02 -1.29318330e-02 -1.00479022e-01 -3.85910720e-02
 -4.02101055e-02 -2.01168191e-02 -9.67080668e-02 -1.88215896e-02
 -9.20955930e-03  6.08900450e-02 -9.28485245e-02  9.75184813e-02
  3.86666544e-02 -3.68803963e-02  3.39471363e-02 -1.04226423e-02
  4.60217595e-02  1.60625391e-02 -9.63037834e-03 -3.49528864e-02
 -1.46012744e-02  3.90401706e-02  1.76329520e-02  7.03792833e-03
 -5.63391633e-02 -1.32602736e-01  2.39239149e-02  1.49508407e-02
  3.06219999e-02  3.14111933e-02 -2.00092401e-02  4.98024523e-02
 -2.98892297e-02  5.22810854e-02  5.35754822e-02 -2.84903054e-03
 -7.70001858e-02 -4.58070226e-02  3.90172638e-02 -1.68384556e-02
 -6.91072550e-03  4.59188111e-02  3.17391916e-03 -4.5528

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Example: Calculate cosine similarity between two sentences
cosine_similarity(
    [model.encode("Easy")],
    [model.encode("Very easy")]
)


array([[0.7629291]], dtype=float32)

## Dimensionality reduction with UMAP

In [43]:
# Define the UMAP model for dimensionality reduction
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')

## Clustering with HDBSCAN

In [ ]:
# Define the HDBSCAN model for clustering
hdbscan_model = HDBSCAN(min_cluster_size=5, min_samples=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

## c-TF-IDF

In [49]:
# Define the CountVectorizer and ClassTfidfTransformer for topic representation
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english') # BI-grams and unigrams, remove English stop words
ctfidf_model = ClassTfidfTransformer()

## Representation model

In [40]:
representation_model = KeyBERTInspired()

## Build BERTopic model

In [50]:
topic_model = BERTopic(
                      embedding_model=model,
                      umap_model=umap_model,
                      hdbscan_model=hdbscan_model,
                      vectorizer_model=vectorizer_model,
                      ctfidf_model=ctfidf_model,
                      representation_model=representation_model,
                      calculate_probabilities=True,
                      verbose=True
                      )

## Fit model

In [59]:
topics, probabilities = topic_model.fit_transform(Q2_list)

2026-06-08 16:21:51,357 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 3/3 [00:07<00:00,  2.63s/it]
2026-06-08 16:21:59,264 - BERTopic - Embedding - Completed ✓
2026-06-08 16:21:59,265 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-08 16:22:10,744 - BERTopic - Dimensionality - Completed ✓
2026-06-08 16:22:10,746 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-08 16:22:10,756 - BERTopic - Cluster - Completed ✓
2026-06-08 16:22:10,759 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-08 16:22:25,313 - BERTopic - Representation - Completed ✓


## Review outputs

In [60]:
# Visualise documents
fig = topic_model.visualize_documents(docs=Q2_list, topics=topics, embeddings=embeddings_Q2)
fig.show()

In [61]:
# Visualse topics
topic_model.visualize_topics()

In [62]:
# Get topic overview
topic_info = topic_model.get_topic_info()
print(topic_info)

for topic_id in topic_info.Topic:
    print(f"Topic {topic_id}:")
    print(topic_model.get_topic(topic_id))

   Topic  Count                                               Name  \
0     -1     10  -1_access community services_ooh services info...   
1      0     42  0_appointments_services_appointments patients_...   
2      1     11  1_hours services_hours services alternatives_s...   
3      2      9  2_services_avoided night shift_gp surgeries op...   

                                      Representation  \
0  [access community services, ooh services info,...   
1  [appointments, services, appointments patients...   
2  [hours services, hours services alternatives, ...   
3  [services, avoided night shift, gp surgeries o...   

                                 Representative_Docs  
0  [TIA pathway!!! Single point of access for com...  
1  [SDEC to accept pt’s. Quicker access to a doct...  
2  [More services out of hours to normal gp hours...  
3  [24/7 Opal, easier intergration of alternative...  
Topic -1:
[('access community services', np.float32(0.46595672)), ('ooh services info', np.fl

In [63]:
# Inspect keywords for the first topic
topic_model.get_topic(0)

[('appointments', np.float32(0.4426271)),
 ('services', np.float32(0.42615628)),
 ('appointments patients', np.float32(0.42157674)),
 ('health services', np.float32(0.39761093)),
 ('mental health services', np.float32(0.3851427)),
 ('referrals', np.float32(0.3664286)),
 ('clinics', np.float32(0.35193235)),
 ('sdec', np.float32(0.34949324)),
 ('direct access', np.float32(0.347338)),
 ('availability', np.float32(0.34697807))]

In [64]:
topic_model.visualize_barchart(top_n_topics=5)

In [65]:
# Generate more meaningful topic labels

topic_model.generate_topic_labels()

['-1_access community services_ooh services info_community services',
 '0_appointments_services_appointments patients',
 '1_hours services_hours services alternatives_services hours normal',
 '2_services_avoided night shift_gp surgeries open']